# 01 — Data Audit

**Obiettivo:** Fotografare lo stato reale dei dati prima di qualunque trasformazione.

Rispondiamo a tre domande fondamentali:
1. Quante informazioni sono effettivamente presenti (vs. missing)?
2. I dati hanno senso (valori anomali, formati incoerenti)?
3. Le due tabelle sono collegate correttamente?

**Output:** `audit_report.txt` con un sommario testuale, più un'ispezione visiva completa.

---
**Dataset:**
- `aziende_principale.csv` — 5370 aziende, 8 colonne
- `aziende_funding_rounds.csv` — 21538 righe, 17 colonne
- Chiave di join: `URL`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import re
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 60)

# ── percorsi ──────────────────────────────────────────────────────────────────
DATA_DIR = Path('../')   # modifica se necessario
COMP_PATH   = DATA_DIR / 'aziende_principale.csv'
ROUNDS_PATH = DATA_DIR / 'aziende_funding_rounds.csv'
OUT_DIR = Path('.')

print('Pandas:', pd.__version__)
print('NumPy: ', np.__version__)

In [ ]:
# ── caricamento ───────────────────────────────────────────────────────────────
comp   = pd.read_csv(COMP_PATH)
rounds = pd.read_csv(ROUNDS_PATH)

print(f'Companies  : {comp.shape[0]:,} righe  × {comp.shape[1]} colonne')
print(f'Rounds     : {rounds.shape[0]:,} righe  × {rounds.shape[1]} colonne')

## 1 · Struttura delle tabelle

In [ ]:
print('=== COMPANIES — dtypes ===')
print(comp.dtypes.to_string())
print()
print('=== ROUNDS — dtypes ===')
print(rounds.dtypes.to_string())

In [ ]:
# Anteprima companies
comp.head(5)

In [ ]:
# Anteprima rounds (trasposta per leggibilità)
rounds.head(5).T

## 2 · Copertura dei dati (missing values)

In [ ]:
def missing_report(df, label):
    """Restituisce una tabella con conteggio e % di valori mancanti."""
    null_count = df.isnull().sum()
    null_pct   = null_count / len(df) * 100
    report = pd.DataFrame({
        'non_null': len(df) - null_count,
        'missing' : null_count,
        'missing_%': null_pct.round(1)
    })
    report['affidabilità'] = pd.cut(
        100 - report['missing_%'],
        bins=[0, 50, 80, 95, 100],
        labels=['❌ scarsa', '⚠️ media', '✅ buona', '✅✅ ottima']
    )
    print(f'\n--- {label} ({len(df):,} righe) ---')
    return report.sort_values('missing_%', ascending=False)

comp_miss   = missing_report(comp,   'COMPANIES')
rounds_miss = missing_report(rounds, 'ROUNDS')

print('\n=== COMPANIES ===')
display(comp_miss)

print('\n=== ROUNDS ===')
display(rounds_miss)

In [ ]:
# ── visualizzazione copertura ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, miss, title in zip(
    axes,
    [comp_miss, rounds_miss],
    ['Companies', 'Rounds']
):
    coverage = 100 - miss['missing_%']
    colors = ['#2ecc71' if v >= 95 else '#f39c12' if v >= 80 else '#e74c3c'
              for v in coverage]
    bars = ax.barh(coverage.index, coverage.values, color=colors, edgecolor='white')
    ax.set_xlim(0, 105)
    ax.axvline(80, color='orange', linestyle='--', alpha=0.6, linewidth=1)
    ax.axvline(95, color='green',  linestyle='--', alpha=0.6, linewidth=1)
    ax.set_xlabel('Copertura (%)')
    ax.set_title(f'{title} — Copertura per colonna')
    for bar, val in zip(bars, coverage.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.0f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'audit_coverage.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura salvata: audit_coverage.png')

### Nota metodologica sul Total Funding

> **`Total Funding` è al 100% mancante** nel dataset companies — il campo non è
> stato popolato dallo scraper. Usiamo i round di finanziamento per ricostruire
> il totale aggregato durante il cleaning. Non è un dato perso, è
> semplicemente da derivare.

## 3 · Qualità della chiave di join

In [ ]:
urls_comp   = set(comp['URL'])
urls_rounds = set(rounds['URL'])

only_comp   = urls_comp   - urls_rounds
only_rounds = urls_rounds - urls_comp
both        = urls_comp   & urls_rounds

print(f'URL unici in companies  : {len(urls_comp):,}')
print(f'URL unici in rounds     : {len(urls_rounds):,}')
print(f'URL in entrambe         : {len(both):,}  ← copertura perfetta')
print(f'Solo in companies       : {len(only_comp):,}')
print(f'Solo in rounds          : {len(only_rounds):,}')

In [ ]:
# Duplicati di URL
dup_comp   = comp['URL'].duplicated().sum()
dup_rounds = rounds.duplicated(subset=['URL','Funding Date','Round Name']).sum()

print(f'URL duplicati in companies           : {dup_comp}')
print(f'(URL + Data + Round) duplicati nei rounds: {dup_rounds}')

## 4 · Ispezione Companies

In [ ]:
# ── Nome azienda: presenza del pattern Forge ──────────────────────────────────
# Lo scraper ha estratto i titoli HTML anziché i nomi puliti
has_forge = comp['Company'].str.contains('- Forge', na=False)
print(f'Nomi con suffisso Forge: {has_forge.sum():,} / {len(comp):,}')
print()
print('Esempi prima del cleaning:')
comp['Company'].head(5).tolist()

In [ ]:
# ── Anno di fondazione ────────────────────────────────────────────────────────
print('=== Founded — statistiche ===')
print(comp['Founded'].describe())
print()

# Valori anomali
print('Aziende fondate prima del 1990 (possibili errori di scraping):')
old = comp[comp['Founded'] < 1990][['Company','Founded']].copy()
old['Company'] = old['Company'].str.extract(r'(.+?)\s+Stock', expand=False)
print(old.to_string(index=False))

print()
print('Aziende con Founded = 1900 (placeholder di scraping):')
print(comp[comp['Founded'] == 1900]['URL'].tolist())

In [ ]:
# ── Distribuzione anno fondazione ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
founded = comp['Founded'].dropna()
founded = founded[(founded >= 1990) & (founded <= 2026)]
ax.hist(founded, bins=range(1990, 2027), color='#3498db', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Anno di fondazione')
ax.set_ylabel('Numero aziende')
ax.set_title('Distribuzione anno di fondazione (1990–2026)')
ax.axvline(2010, color='red', linestyle='--', alpha=0.5, label='2010')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'audit_founded.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Settori ───────────────────────────────────────────────────────────────────
print('=== Distribuzione Sector ===')
print(comp['Sector'].value_counts().to_string())
print()
print('=== Top 20 Subsector ===')
print(comp['Subsector'].value_counts().head(20).to_string())

In [ ]:
# ── Sede ─────────────────────────────────────────────────────────────────────
def extract_country(hq):
    if pd.isna(hq): return np.nan
    parts = str(hq).strip().split(',')
    return parts[-1].strip()

countries = comp['Headquarters'].apply(extract_country)
print('=== Top 20 paesi ===')
print(countries.value_counts().head(20).to_string())
print()

us_states = comp['Headquarters'].dropna()
us_only = us_states[us_states.str.contains('United States')]
def extract_state(hq):
    m = re.search(r',\s*([A-Z]{2}),\s*United States', hq)
    return m.group(1) if m else 'Other'
states = us_only.apply(extract_state)
print('=== Top 10 stati USA ===')
print(states.value_counts().head(10).to_string())

## 5 · Ispezione Rounds

In [ ]:
# ── Righe senza dati reali (piccole aziende, template vuoto) ──────────────────
# Lo scraper usa un template di fallback per aziende senza round dettagliati.
# Queste righe hanno Funding Date = NaN.
real   = rounds.dropna(subset=['Funding Date'])
empty  = rounds[rounds['Funding Date'].isna()]

print(f'Rounds con dati reali   : {len(real):,}')
print(f'Righe placeholder vuote : {len(empty):,}')
print(f'Aziende con ≥1 round    : {real["URL"].nunique():,}')
print(f'Aziende senza round     : {empty["URL"].nunique():,}')

In [ ]:
# ── Distribuzione Round Name ──────────────────────────────────────────────────
print('=== Round Name — top 30 ===')
print(real['Round Name'].value_counts().head(30).to_string())

In [ ]:
# ── Format dei valori monetari ────────────────────────────────────────────────
print('Campioni Amount Raised:')
print(real['Amount Raised'].dropna().head(15).tolist())
print()
print('Campioni Post-Money Valuation:')
print(real['Post-Money Valuation'].dropna().head(15).tolist())
print()
print('Campioni Price Per Share:')
print(real['Price Per Share (Overview)'].dropna().head(15).tolist())

In [ ]:
# ── Formato date ─────────────────────────────────────────────────────────────
print('Campioni Funding Date:')
print(real['Funding Date'].dropna().head(15).tolist())

# Tentativo di parsing
dates_parsed = pd.to_datetime(real['Funding Date'], format='%m/%d/%Y', errors='coerce')
ok = dates_parsed.notna().sum()
print(f'\nDate parsate con successo: {ok:,} / {len(real):,} ({ok/len(real)*100:.1f}%)')

In [ ]:
# ── Round per azienda ─────────────────────────────────────────────────────────
rpc = real.groupby('URL').size().rename('n_rounds')
print('=== Round per azienda ===')
print(rpc.describe())
print()
print('Aziende con più round:')
top = rpc.nlargest(10).reset_index()
top['Company'] = top['URL'].str.extract(r'forgeglobal\.com/(.+?)_stock/', expand=False)
print(top[['Company','n_rounds']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
rpc.clip(upper=20).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='#9b59b6', edgecolor='white'
)
ax.set_xlabel('Numero di round (20+ accorpati)')
ax.set_ylabel('Numero di aziende')
ax.set_title('Distribuzione round per azienda')
plt.tight_layout()
plt.savefig(OUT_DIR / 'audit_rounds_per_company.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Termini preferenziali (dati strutturati da securities) ────────────────────
print('=== Liquidation Pref As Multiplier ===')
print(real['Liquidation Pref As Multiplier'].value_counts().head(10).to_string())
print()
print('=== Participating ===')
print(real['Participating'].value_counts().to_string())
print()
print('=== Cumulative ===')
print(real['Cumulative'].value_counts().to_string())

## 6 · Anomalie specifiche da gestire nel cleaning

In [ ]:
print('=== ANOMALIE IDENTIFICATE ===')
print()

# 1. Nomi aziende
forge_names = comp['Company'].str.contains('- Forge', na=False).sum()
print(f'1. Nomi con rumore Forge HTML: {forge_names:,} / {len(comp):,}')

# 2. Founded = 1900 (placeholder scraping)
f1900 = (comp['Founded'] == 1900).sum()
print(f'2. Founded = 1900 (placeholder): {f1900}')

# 3. Founded > 2024
f_future = (comp['Founded'] > 2024).sum()
print(f'3. Founded > 2024: {f_future} (aziende molto recenti, ok)')

# 4. Amount Raised = '--'
dash_amount = (real['Amount Raised'] == '--').sum()
print(f'4. Amount Raised = "--": {dash_amount} round')

# 5. Sector = 'Missing'
sect_missing = (comp['Sector'] == 'Missing').sum()
print(f'5. Sector = "Missing": {sect_missing} aziende')

# 6. Total Funding: completamente vuoto
tf_null = comp['Total Funding'].isna().sum()
print(f'6. Total Funding null: {tf_null:,} ({tf_null/len(comp)*100:.0f}%) — da ricalcolare da rounds')

# 7. Shares Outstanding con valore '1' (anomalo)
shares_one = (real['Shares Outstanding'] == '1').sum()
print(f'7. Shares Outstanding = "1": {shares_one} round (possibile errore scraping)')

## 7 · Analisi investor coverage

In [ ]:
# Quante aziende hanno almeno un investitore noto?
inv_in_comp   = comp['Investors'].notna().sum()
inv_in_rounds = real['Key Investors'].notna().sum()
inv_in_ov     = real['Key Investors (Overview)'].notna().sum()

print(f'Aziende con investitori in companies  : {inv_in_comp:,} / {len(comp):,}')
print(f'Round con Key Investors in rounds     : {inv_in_rounds:,} / {len(real):,}')
print(f'Round con Key Investors (Overview)    : {inv_in_ov:,} / {len(real):,}')

# Sample investor list
print('\nCampione investitori (companies):')
print(comp['Investors'].dropna().head(5).tolist())

## 8 · Report finale

In [ ]:
report = """
==============================================================
FORGE GLOBAL VC DATASET — DATA AUDIT REPORT
==============================================================

DIMENSIONI
  Companies  : 5370 aziende  ×  8 colonne
  Rounds     : 21538 righe   × 17 colonne
  Join key   : URL (copertura 100%, nessun orfano)

QUALITÀ COMPANIES
  ✅✅ URL, Company, Sector, Subsector : 100% presenti
  ✅✅ Founded                         : 99.9% (3 missing, 3 con 1900 come placeholder)
  ✅✅ Headquarters                    : 99.5% (27 missing)
  ⚠️  Investors                       : 52.8% presenti
  ❌  Total Funding                    : 0% — da ricalcolare dai rounds
  ⚠️  Company names                    : contengono rumore HTML tipo
                                        "Invest and Sell X Stock - Forge" → estrarre X

QUALITÀ ROUNDS
  ✅✅ URL                             : 100%
  ⚠️  Righe senza dati (placeholder)  : 2282 / 21538 = 10.6%
                                        (aziende senza round dettagliati su Forge)
  ✅✅ Round con dati reali            : 19256
  ✅✅ Funding Date                    : 100% parseable MM/DD/YYYY
  ✅  Amount Raised                   : 99.9% parseable ($MM/$B)
  ✅✅ Post-Money Valuation            : 100% parseable
  ⚠️  Key Investors                   : 89.4% presenti nei round reali

ANOMALIE DA GESTIRE
  1. Company name: stripping del template HTML Forge
  2. Founded = 1900: trattare come NaN (placeholder scraping)
  3. Amount Raised = '--': trattare come NaN
  4. Sector/Subsector = 'Missing': trattare come NaN
  5. Total Funding: derivare da sum(Amount Raised) per azienda
  6. Shares Outstanding = '1': possibile errore, da escludere in analisi specifiche

ANALISI AFFIDABILI CON QUESTI DATI
  ✅ Distribuzione per settore e subsettore
  ✅ Analisi round (tipo, importi, valutazioni, timing)
  ✅ Analisi temporale (anno fondazione, anno round)
  ✅ Analisi geografica (paese, stato USA)
  ✅ Network investitori (limitato al 52% aziende)
  ✅ Termini preferenziali (liquidation pref, partecipazione)
  ⚠️ Totale funding per azienda (derivato, non nativo)
  ❌ Analisi di sopravvivenza/successo (survivorship bias)

==============================================================
"""

print(report)
with open(OUT_DIR / 'audit_report.txt', 'w') as f:
    f.write(report)
print('Salvato: audit_report.txt')